# Industrial AI Data Pipeline
### Databricks / PySpark / Delta Lake / Anomaly Detection

**Purpose:** End-to-end pipeline demonstration on synthetic industrial sensor data.  
**Pipeline:** Raw CSV → PySpark ingestion → data-quality inspection → cleaning → validation → Delta table → feature engineering → anomaly detection → Matplotlib visualisation → final quality summary.

**Dataset:** Synthetic. Four machines (M-101–M-104), 10-minute sensor readings, August 2026.  
Deliberate quality problems injected: missing values, duplicate rows, out-of-range readings, anomalous spikes.

---
**Relation to EU AI Act research:** This project can serve as the industrial case study for data-quality and provenance evidence under Annex IV documentation requirements.

## 0. Configuration

Set paths and thresholds here. All pipeline stages read from this cell.

### Databricks setup — Unity Catalog Volumes

Use these steps when public DBFS root (`/FileStore/`) is disabled in your workspace.

**1. Create the volume (run once in a SQL cell or the SQL editor):**
```sql
CREATE CATALOG IF NOT EXISTS main;
CREATE SCHEMA IF NOT EXISTS main.default;
CREATE VOLUME IF NOT EXISTS main.default.industrial_pipeline;
```

**2. Upload the CSV to the volume:**
- UI: Catalog → Volumes → `industrial_pipeline` → Upload → select `data/raw/sensor_readings.csv`
- CLI: `databricks fs cp data/raw/sensor_readings.csv /Volumes/main/default/industrial_pipeline/sensor_readings.csv`

**3. Adjust `CATALOG`, `SCHEMA`, and `VOLUME` in the next cell if you used different names.**

Delta output tables will be written under the same volume automatically.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Unity Catalog Volume paths — works when public DBFS root (/FileStore/) is disabled.
# Adjust CATALOG, SCHEMA, VOLUME to match your workspace.
CATALOG = "main"
SCHEMA  = "default"
VOLUME  = "industrial_pipeline"

INPUT      = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/sensor_readings.csv"
DELTA_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/delta/cleaned"
FEAT_PATH  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/delta/features"

# Normal operating envelopes (used for cleaning and feature engineering)
TEMP_NORMAL_MEAN  = 72.0
TEMP_NORMAL_STD   =  5.0
PRESSURE_NOMINAL  =  5.2
VIBRATION_RISK_THRESHOLD = 4.0

print("Configuration loaded.")
print(f"Input CSV  : {INPUT}")
print(f"Delta clean: {DELTA_PATH}")
print(f"Delta feat : {FEAT_PATH}")

## 1. Ingest raw CSV with PySpark

We use PySpark's CSV reader with schema inference. In a production pipeline the schema would be declared explicitly — schema drift is a common source of silent failures.

In [ ]:
raw = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(INPUT)
)

raw.printSchema()
display(raw.limit(10))

## 2. Data-quality inspection

Before touching the data we record its baseline state. These metrics become provenance evidence: we can demonstrate what the data looked like on arrival, which matters for EU AI Act Annex IV documentation.

In [ ]:
raw_count = raw.count()
raw_distinct = raw.dropDuplicates().count()
duplicate_count = raw_count - raw_distinct

print(f"Total rows      : {raw_count:,}")
print(f"Distinct rows   : {raw_distinct:,}")
print(f"Duplicate rows  : {duplicate_count:,}")
print()

# Null counts per column
print("Null counts per column:")
raw.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]
).show()

# Descriptive statistics
print("Descriptive statistics:")
display(raw.describe())

### 2a. Out-of-range inspection

Physical sensor limits define valid ranges. Readings outside these ranges indicate faulty sensors, transmission errors, or deliberate injection events — all of which must be detected before the data enters a model.

In [ ]:
invalid_temp     = raw.filter(~F.col("temperature_c").between(-20, 120)).count()
invalid_pressure = raw.filter(~F.col("pressure_bar").between(0, 12)).count()
invalid_vib      = raw.filter(~F.col("vibration_mm_s").between(0, 20)).count()
null_machine     = raw.filter(F.col("machine_id").isNull()).count()
invalid_ts       = raw.filter(F.to_timestamp("timestamp").isNull()).count()

print("Out-of-range counts (pre-cleaning):")
print(f"  temperature_c outside [-20, 120] : {invalid_temp}")
print(f"  pressure_bar  outside [0, 12]    : {invalid_pressure}")
print(f"  vibration_mm_s outside [0, 20]   : {invalid_vib}")
print(f"  null machine_id                  : {null_machine}")
print(f"  unparseable timestamp            : {invalid_ts}")

known_machines = {"M-101", "M-102", "M-103", "M-104"}
unexpected_machines = (
    raw.filter(F.col("machine_id").isNotNull())
       .filter(~F.col("machine_id").isin(*known_machines))
       .count()
)
print(f"  unexpected machine IDs           : {unexpected_machines}")

## 3. Cleaning

Cleaning strategy:
- Parse timestamp column to proper type.
- Drop full duplicate rows.
- Remove rows with null `timestamp` or `machine_id` (cannot be attributed to a machine or time-window).
- Filter out physically implausible sensor readings.
- Fill remaining nulls in sensor columns with machine-appropriate medians (approximated by global medians for this demo).

In [ ]:
cleaned = (
    raw
    # Parse timestamp
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    # Remove exact duplicates
    .dropDuplicates()
    # Drop rows where machine identity or time is unknown
    .na.drop(subset=["timestamp", "machine_id"])
    # Remove rows with unknown machine IDs
    .filter(F.col("machine_id").isin(*["M-101", "M-102", "M-103", "M-104"]))
    # Physical range filters — values outside these are instrumentation errors
    .filter(F.col("temperature_c").isNull()   | F.col("temperature_c").between(-20, 120))
    .filter(F.col("pressure_bar").isNull()    | F.col("pressure_bar").between(0, 12))
    .filter(F.col("vibration_mm_s").isNull()  | F.col("vibration_mm_s").between(0, 20))
    # Impute remaining nulls with approximate fleet-wide medians
    .fillna({
        "temperature_c"  : 72.0,
        "pressure_bar"   :  5.2,
        "vibration_mm_s" :  2.0,
        "energy_kwh"     : 48.0,
    })
)

cleaned_count = cleaned.count()
print(f"Rows before cleaning : {raw_count:,}")
print(f"Rows after cleaning  : {cleaned_count:,}")
print(f"Rows removed         : {raw_count - cleaned_count:,}")
display(cleaned.limit(10))

## 4. Post-cleaning validation

Assertions serve as pipeline gates. If any assertion fails, the run stops with a clear error rather than silently passing bad data into the Delta table.

In [ ]:
# No nulls should remain in critical columns
remaining_nulls = cleaned.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cleaned.columns]
).first()

for col_name in ["timestamp", "machine_id", "temperature_c", "pressure_bar", "vibration_mm_s", "energy_kwh"]:
    assert remaining_nulls[col_name] == 0, f"Unexpected nulls in {col_name}: {remaining_nulls[col_name]}"

# All four expected machines present
machine_count = cleaned.select("machine_id").distinct().count()
assert machine_count == 4, f"Expected 4 machines, found {machine_count}"

# No duplicates remain
remaining_dupes = cleaned.count() - cleaned.dropDuplicates().count()
assert remaining_dupes == 0, f"Duplicates still present: {remaining_dupes}"

# Sensor values within valid bounds
assert cleaned.filter(~F.col("temperature_c").between(-20, 120)).count() == 0, "Temperature out of range"
assert cleaned.filter(~F.col("pressure_bar").between(0, 12)).count() == 0, "Pressure out of range"
assert cleaned.filter(~F.col("vibration_mm_s").between(0, 20)).count() == 0, "Vibration out of range"

print("All validation assertions passed.")
print(f"Clean dataset: {cleaned_count:,} rows, 4 machines, 0 nulls, 0 duplicates.")

## 5. Write to Delta table

Delta Lake adds ACID transactions, schema enforcement, and time-travel to Parquet storage. Writing the cleaned data as Delta creates the first durable, queryable layer of the pipeline — equivalent to the Silver layer in a medallion architecture.

In [ ]:
# Write cleaned dataset as Delta (overwrite for idempotency)
cleaned.write.format("delta").mode("overwrite").save(DELTA_PATH)

# Read back to confirm round-trip integrity
delta_df = spark.read.format("delta").load(DELTA_PATH)
delta_row_count = delta_df.count()

assert delta_row_count == cleaned_count, (
    f"Delta row count mismatch: wrote {cleaned_count}, read back {delta_row_count}"
)
print(f"Delta table written and verified: {delta_row_count:,} rows at {DELTA_PATH}")

# Delta table history (demonstrates lineage / audit trail)
display(spark.sql(f"DESCRIBE HISTORY delta.`{DELTA_PATH}`"))

## 6. Feature engineering

Four domain-relevant features derived from the cleaned sensor readings:

| Feature | Definition | Industrial meaning |
|---|---|---|
| `temp_z_proxy` | (temperature − 72) / 5 | Z-score proxy: standard deviations from fleet mean |
| `pressure_deviation` | \|pressure − 5.2\| | Absolute deviation from nominal operating pressure |
| `vibration_risk` | 1 if vibration ≥ 4 mm/s else 0 | Binary flag for elevated vibration (bearing wear indicator) |
| `energy_per_vibration` | energy_kwh / (vibration + 0.01) | Efficiency ratio: high energy with low vibration is normal; inverse signals mechanical problems |

In [ ]:
features = (
    delta_df
    .withColumn(
        "temp_z_proxy",
        (F.col("temperature_c") - TEMP_NORMAL_MEAN) / TEMP_NORMAL_STD
    )
    .withColumn(
        "pressure_deviation",
        F.abs(F.col("pressure_bar") - PRESSURE_NOMINAL)
    )
    .withColumn(
        "vibration_risk",
        F.when(F.col("vibration_mm_s") >= VIBRATION_RISK_THRESHOLD, 1).otherwise(0)
    )
    .withColumn(
        "energy_per_vibration",
        F.col("energy_kwh") / (F.col("vibration_mm_s") + F.lit(0.01))
    )
)

print("Feature columns added:")
print("  temp_z_proxy, pressure_deviation, vibration_risk, energy_per_vibration")
display(features.select(
    "timestamp", "machine_id",
    "temperature_c", "temp_z_proxy",
    "pressure_bar", "pressure_deviation",
    "vibration_mm_s", "vibration_risk",
    "energy_kwh", "energy_per_vibration"
).limit(10))

## 7. Basic anomaly detection

Rule-based anomaly detection: a reading is flagged as anomalous if any of three independent conditions are met.

| Condition | Threshold | Rationale |
|---|---|---|
| Temperature Z-score | > 3σ | Statistical: more than 3 standard deviations from normal |
| Pressure deviation | > 1.5 bar | Domain: excessive deviation from nominal 5.2 bar |
| Vibration flag | vibration_risk = 1 | Domain: elevated vibration indicates bearing wear |

This is deliberately simple. A production system would layer IsolationForest or LSTM-based approaches on top of these rule signals.

In [ ]:
anomalies = features.withColumn(
    "anomaly",
    F.when(
        (F.abs(F.col("temp_z_proxy")) > 3.0)
        | (F.col("pressure_deviation") > 1.5)
        | (F.col("vibration_risk") == 1),
        1
    ).otherwise(0)
)

total_anomalies = anomalies.filter(F.col("anomaly") == 1).count()
anomaly_rate = total_anomalies / anomalies.count() * 100

print(f"Total anomalies detected : {total_anomalies:,}")
print(f"Anomaly rate             : {anomaly_rate:.1f}%")
print()
print("Anomalies per machine:")
display(
    anomalies
    .groupBy("machine_id")
    .agg(
        F.count("*").alias("total_readings"),
        F.sum("anomaly").alias("anomaly_count"),
        (F.sum("anomaly") / F.count("*") * 100).alias("anomaly_rate_pct")
    )
    .orderBy(F.desc("anomaly_count"))
)

### 7a. Save feature + anomaly layer

In [ ]:
anomalies.write.format("delta").mode("overwrite").save(FEAT_PATH)
print(f"Features + anomaly scores saved to {FEAT_PATH}")

## 8. Matplotlib visualisation

Two plots:
1. **Temperature time series** for each machine — shows normal operating band and flagged anomalies.
2. **Anomaly distribution by machine** — bar chart of anomaly counts.

We sample 1,000 rows for the time-series plot to avoid Spark-to-driver memory issues in Databricks.

In [ ]:
plot_df = (
    anomalies
    .select("timestamp", "machine_id", "temperature_c", "vibration_mm_s", "anomaly")
    .orderBy("timestamp")
    .limit(1000)
    .toPandas()
)

# --- Plot 1: Temperature time series with anomaly highlights ---
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

machines = sorted(plot_df["machine_id"].dropna().unique())
colors   = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

ax1 = axes[0]
for machine, color in zip(machines, colors):
    m_df = plot_df[plot_df["machine_id"] == machine].sort_values("timestamp")
    ax1.plot(m_df["timestamp"], m_df["temperature_c"], label=machine, color=color, alpha=0.7, lw=1)
    # Highlight anomalous readings
    anom = m_df[m_df["anomaly"] == 1]
    ax1.scatter(anom["timestamp"], anom["temperature_c"], color=color, s=25, zorder=5, marker="x")

ax1.axhspan(55, 90, alpha=0.05, color="green", label="Normal band")
ax1.set_ylabel("Temperature (°C)")
ax1.set_title("Industrial Sensor: Temperature Time Series (× = anomaly)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(True, alpha=0.3)

# --- Plot 2: Vibration time series ---
ax2 = axes[1]
for machine, color in zip(machines, colors):
    m_df = plot_df[plot_df["machine_id"] == machine].sort_values("timestamp")
    ax2.plot(m_df["timestamp"], m_df["vibration_mm_s"], label=machine, color=color, alpha=0.7, lw=1)

ax2.axhline(VIBRATION_RISK_THRESHOLD, color="red", ls="--", lw=1, label=f"Risk threshold ({VIBRATION_RISK_THRESHOLD} mm/s)")
ax2.set_ylabel("Vibration (mm/s)")
ax2.set_xlabel("Time")
ax2.set_title("Vibration by Machine")
ax2.legend(loc="upper right", fontsize=8)
ax2.grid(True, alpha=0.3)

plt.gcf().autofmt_xdate()
plt.tight_layout()
plt.show()

# --- Plot 3: Anomaly counts per machine ---
anomaly_summary = (
    anomalies
    .groupBy("machine_id")
    .agg(F.sum("anomaly").alias("anomaly_count"))
    .orderBy("machine_id")
    .toPandas()
)

fig2, ax3 = plt.subplots(figsize=(7, 4))
ax3.bar(anomaly_summary["machine_id"], anomaly_summary["anomaly_count"], color=colors[:len(machines)])
ax3.set_xlabel("Machine")
ax3.set_ylabel("Anomaly count")
ax3.set_title("Anomalies Detected per Machine")
ax3.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Final data-quality summary

This cell produces the final quality evidence record. In a production pipeline, this would be written to a monitoring table or forwarded to a data observability platform (e.g., Great Expectations, Monte Carlo).

In [ ]:
quality = anomalies.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("machine_id").alias("machines"),
    F.sum("anomaly").alias("anomaly_rows"),
    F.min("timestamp").alias("min_timestamp"),
    F.max("timestamp").alias("max_timestamp"),
    F.sum(F.col("temperature_c").isNull().cast("int")).alias("null_temperature"),
    F.sum(F.col("pressure_bar").isNull().cast("int")).alias("null_pressure"),
    F.sum(F.col("vibration_mm_s").isNull().cast("int")).alias("null_vibration"),
)
display(quality)

q = quality.first()
assert q["total_rows"] > 0,    "Pipeline produced zero output rows"
assert q["machines"]  == 4,    f"Expected 4 machines, got {q['machines']}"
assert q["null_temperature"] == 0, "Nulls remain in temperature_c"
assert q["null_pressure"]    == 0, "Nulls remain in pressure_bar"
assert q["null_vibration"]   == 0, "Nulls remain in vibration_mm_s"

print(f"\nPipeline complete.")
print(f"  Clean rows    : {q['total_rows']:,}")
print(f"  Machines      : {q['machines']}")
print(f"  Anomalies     : {q['anomaly_rows']:,} ({q['anomaly_rows']/q['total_rows']*100:.1f}%)")
print(f"  Time range    : {q['min_timestamp']} → {q['max_timestamp']}")
print(f"  Null checks   : passed")
print(f"  Assertions    : all passed")